<a href="https://colab.research.google.com/github/Aswathi281099/Generative-Artificial-Intelligence/blob/main/AI_TASK_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TASK 13**

In [1]:
!pip install -q transformers datasets accelerate


In [2]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
data = {
    "prompt": ["What is AI?", "What is Python?"],
    "chosen": [
        "AI helps computers perform intelligent tasks.",
        "Python is a programming language."
    ],
    "rejected": [
        "AI is impossible to understand.",
        "Python is computer hardware."
    ]
}

dataset = Dataset.from_dict(data)
print(dataset)

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 2
})


In [4]:
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Trainable policy model
policy_model = AutoModelForCausalLM.from_pretrained(model_name)

# Frozen reference model
reference_model = AutoModelForCausalLM.from_pretrained(model_name)

for p in reference_model.parameters():
    p.requires_grad = False

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

...

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.51MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def log_prob(model, text):
    inputs = tokenizer(text, return_tensors="pt")

    outputs = model(**inputs)

    logits = outputs.logits[:, :-1, :]
    labels = inputs["input_ids"][:, 1:]

    log_probs = torch.log_softmax(logits, dim=-1)

    return log_probs.gather(
        2, labels.unsqueeze(-1)
    ).squeeze(-1).sum()

In [6]:
sample = dataset[0]

prompt = sample["prompt"]
chosen = prompt + " " + sample["chosen"]
rejected = prompt + " " + sample["rejected"]

policy_chosen = log_prob(policy_model, chosen)
policy_rejected = log_prob(policy_model, rejected)

with torch.no_grad():
    ref_chosen = log_prob(reference_model, chosen)
    ref_rejected = log_prob(reference_model, rejected)

beta = 0.1

loss = -torch.log(torch.sigmoid(
    beta * (
        (policy_chosen - ref_chosen)
        - (policy_rejected - ref_rejected)
    )
))

print("DPO Loss:", loss.item())

DPO Loss: 0.6931471824645996


In [7]:
optimizer = torch.optim.AdamW(
    policy_model.parameters(),
    lr=1e-4
)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Policy model updated!")

Policy model updated!


In [8]:
print("DPO training completed successfully!")
print("Chosen and rejected responses were compared.")
print("Policy model was updated using DPO loss.")

DPO training completed successfully!
Chosen and rejected responses were compared.
Policy model was updated using DPO loss.
